# Tunix-Med: Medical Synthetic Data Preparation

This notebook automates the creation of a high-quality medical QA dataset focused on **Cardiology**. It follows a professional synthetic data pipeline used for domain adaptation:

1. **Scrape**: Crawl Wikipedia for specific cardiology pathologies and clinical guidelines.
2. **Generate**: Use `synthetic-data-kit` (vLLM) to produce clinical QA pairs.
3. **Curate**: Filter out low-quality or inaccurate pairs using an LLM judge.
4. **Format**: Convert the data into the structured conversation format required for fine-tuning with **Tunix**.

Set `DEMO = True` to run a quick test on a subset of the data.

In [ ]:
import os
import re
import time
import subprocess
import wikipediaapi
import pandas as pd
from tqdm import tqdm
from datasets import Dataset, DatasetDict

# --- Configuration ---
DEMO = True
MODEL_ENGINE = "Qwen/Qwen2.5-7B-Instruct-AWQ"
NUM_PAIRS_PER_FILE = 30
QUALITY_THRESHOLD = 7.0


def init():
    os.makedirs("data/medical_input", exist_ok=True)
    os.makedirs("data/generated", exist_ok=True)
    os.makedirs("data/output", exist_ok=True)
    os.makedirs("data/final", exist_ok=True)


init()

## 1. Domain-Specific Scraping
We target the specific pathologies inquired about in our baseline evaluation.

In [ ]:
def clean_string(input_string):
    cleaned_string = " ".join(input_string.split())
    cleaned_string = re.sub(r"\{.*?\}|\}", "", cleaned_string)
    return cleaned_string


def get_wikipedia_content(titles):
    wiki_wiki = wikipediaapi.Wikipedia("MedicalBot (medbot@example.com)", "en")
    extracted_texts = []
    print("- Fetching medical articles:")
    for title in tqdm(titles):
        page = wiki_wiki.page(title)
        if page.exists():
            extracted_texts.append(page.title + " : " + clean_string(page.summary))
            for section in page.sections:
                if len(section.text) > 200:
                    extracted_texts.append(
                        page.title
                        + " ("
                        + section.title
                        + ") : "
                        + clean_string(section.text)
                    )
    return extracted_texts


cardiology_topics = [
    "Hypertension",
    "Acute coronary syndrome",
    "Myocardial infarction",
    "Stable angina",
    "Unstable angina",
    "Coronary artery disease",
    "Coronary artery spasm",
    "TIMI risk score",
    "Percutaneous coronary intervention",
    "Coronary artery bypass surgery",
    "Heart failure",
    "Heart failure with preserved ejection fraction",
    "Dilated cardiomyopathy",
    "Hypertrophic cardiomyopathy",
    "Restrictive cardiomyopathy",
    "Takotsubo cardiomyopathy",
    "Cardiac resynchronization therapy",
    "Atrial fibrillation",
    "Atrial flutter",
    "Supraventricular tachycardia",
    "Ventricular tachycardia",
    "Ventricular fibrillation",
    "Wolff-Parkinson-White syndrome",
    "Long QT syndrome",
    "Brugada syndrome",
    "Sick sinus syndrome",
    "Atrioventricular block",
    "Cardiac pacemaker",
    "Implantable cardioverter-defibrillator",
    "Aortic stenosis",
    "Aortic regurgitation",
    "Mitral stenosis",
    "Mitral regurgitation",
    "Mitral valve prolapse",
    "Tricuspid regurgitation",
    "Infective endocarditis",
    "Transcatheter aortic valve replacement",
    "Pericarditis",
    "Cardiac tamponade",
    "Beck's triad (cardiac tamponade)",
    "Myocarditis",
    "Constrictive pericarditis",
    "Aortic aneurysm",
    "Aortic dissection",
    "Peripheral artery disease",
    "Deep vein thrombosis",
    "Pulmonary embolism",
    "Dyslipidemia",
    "Familial hypercholesterolemia",
    "Atherosclerosis",
    "Metabolic syndrome",
    "Congenital heart disease",
    "Atrial septal defect",
    "Ventricular septal defect",
    "Tetralogy of Fallot",
    "Patent ductus arteriosus",
    "Pulmonary hypertension",
    "Cor pulmonale",
    "Electrocardiography",
    "Echocardiography",
    "Cardiac stress test",
    "Cardiac catheterization",
    "Holter monitor",
    "Troponin",
    "B-type natriuretic peptide",
    "Beta blocker",
    "ACE inhibitor",
    "Calcium channel blocker",
    "Statin",
    "Anticoagulant",
    "Antiarrhythmic agent",
    "Diuretic",
    "Digoxin",
    "Nitrate (pharmacology)",
]

if DEMO:
    cardiology_topics = cardiology_topics[:5]

extracted_texts = get_wikipedia_content(cardiology_topics)
for i, text in enumerate(extracted_texts):
    with open(os.path.join("data/medical_input", f"med_{i}.txt"), "w") as f:
        f.write(text)
    if DEMO and i > 15:
        break

- Fetching medical articles:


100%|██████████| 5/5 [00:02<00:00,  1.73it/s]


## 2. Configuration & vLLM Server

In [ ]:
def start_vllm():
    env = os.environ.copy()
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["VLLM_MOE_KERNEL_BACKEND"] = "triton"
    env["VLLM_DISABLED_KERNELS"] = "cutlass_moe_mm,cutlass_scaled_mm"

    vllm_log = open("vllm_server.log", "w")
    return subprocess.Popen(
        [
            "vllm",
            "serve",
            MODEL_ENGINE,
            "--port",
            "8000",
            "--enforce-eager",
            "--gpu-memory-utilization",
            "0.25",
            "--max-model-len",
            "8192",
            "--disable-log-stats",  # still valid: silences throughput/KV stats
            "--uvicorn-log-level",
            "error",  # silences HTTP access logs (GET /v1/models etc.)
        ],
        env=env,
        stdout=vllm_log,
        stderr=vllm_log,
    )


vllm_proc = start_vllm()

print("Waiting for vLLM server to be ready...")
while "Starting vLLM server" not in open("vllm_server.log").read():
    time.sleep(2)
print("vLLM server ready!")

Waiting for vLLM server to be ready...
vLLM server ready!


In [ ]:
yaml_content = f"""
# Master configuration for Synthetic Data Kit

paths:
  input:
    txt: "data/medical_input"
  output:
    parsed: "data/output"
    generated: "data/generated"
    cleaned: "data/cleaned"
    final: "data/final"

vllm:
  api_base: "http://localhost:8000/v1"
  port: 8000
  model: "{MODEL_ENGINE}"
  max_retries: 3
  retry_delay: 1.0

ingest:
  default_format: "txt"

generation:
  temperature: 0.4
  top_p: 0.95
  chunk_size: 1022
  overlap: 64
  max_tokens: 1024
  num_pairs: {NUM_PAIRS_PER_FILE}

curate:
  threshold: {QUALITY_THRESHOLD}
  batch_size: 4
  temperature: 0.3

format:
  default: "jsonl"
  include_metadata: true
  pretty_json: true

prompts:
  summary: |
    Summarize the following medical text in 3-5 concise sentences.
    Focus on the key clinical concepts, diagnostic criteria, and treatment highlights.
    Return ONLY the summary text, no preamble.
    Text: {{text}}

  qa_generation: |
    Create {NUM_PAIRS_PER_FILE} question-answer pairs from this medical text for clinical training.
    Focus on diagnostic criteria, treatments, and pathophysiology.
    Output ONLY a valid JSON array, no explanations, no markdown:

    [
      {{{{
        "question": "Question 1?",
        "answer": "Answer 1."
      }}}},
      {{{{
        "question": "Question 2?",
        "answer": "Answer 2."
      }}}}
    ]

    Text: {{text}}

  qa_rating: |
    Rate each of these question-answer pairs for quality and return exactly this JSON format:

    [
      {{{{"question": "same question text", "answer": "same answer text", "rating": n}}}}
    ]

    Where n is a number from 1-10, using this scale:
    1=wrong/nonsensical
    3=vague/incomplete
    5=correct but generic
    7=accurate and useful
    9=insightful and precise
    10=perfect

    DO NOT include any text outside of the JSON array, just return valid JSON:

    {{pairs}}
"""

with open("medical_data_config.yaml", "w") as f:
    f.write(yaml_content)

## 3. Pipeline Execution

In [4]:
input_files = [
    os.path.join("data/medical_input", f)
    for f in os.listdir("data/medical_input")
    if f.endswith(".txt")
]
vllm_log = open("vllm_server.log", "a")

print("Starting generation and curation loop...")
for filepath in tqdm(input_files):
    # 1. Create (Silence output)
    subprocess.run(
        [
            "synthetic-data-kit",
            "-c",
            "medical_data_config.yaml",
            "create",
            filepath,
            "--type",
            "qa",
        ],
        stdout=vllm_log,
        stderr=vllm_log,
    )

    # 2. Curate (Silence output)
    gen_file = os.path.join(
        "data/generated", os.path.basename(filepath).replace(".txt", "_qa_pairs.json")
    )
    if os.path.exists(gen_file):
        subprocess.run(
            [
                "synthetic-data-kit",
                "-c",
                "medical_data_config.yaml",
                "curate",
                gen_file,
            ],
            stdout=vllm_log,
            stderr=vllm_log,
        )

    # 3. Format conversion
    if os.path.exists(gen_file):
        subprocess.run(
            [
                "synthetic-data-kit",
                "-c",
                "medical_data_config.yaml",
                "save-as",
                gen_file,
                "-f",
                "ft",
            ],
            stdout=vllm_log,
            stderr=vllm_log,
        )

vllm_proc.terminate()
vllm_log.close()
print("Generation and Curation complete!")

Starting generation and curation loop...


100%|██████████| 17/17 [42:58<00:00, 151.69s/it]

Generation and Curation complete!


## 4. Final Preparation

In [12]:
SYSTEM_PROMPT = "You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology."


def set_system_prompt(row):
    row["messages"][0]["content"] = SYSTEM_PROMPT
    return row


conversations = pd.concat(
    [pd.read_json(f"data/final/{name}") for name in os.listdir("data/final")]
).reset_index(drop=True)

conversations = conversations.apply(set_system_prompt, axis=1)

In [13]:
complete_dataset = DatasetDict(
    {
        "train": Dataset.from_pandas(conversations),
    }
)

In [14]:
complete_dataset.save_to_disk("data/medical-cardiology-qa")
complete_dataset.push_to_hub("lmassaron/medical-cardiology-qa", private=True)

Saving the dataset (0/1 shards):   0%|          | 0/334 [00:00<?, ? examples/s]

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/lmassaron/medical-cardiology-qa/commit/c213320c04d27456bd00daabb07b1978ba838921', commit_message='Upload dataset', commit_description='', oid='c213320c04d27456bd00daabb07b1978ba838921', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/lmassaron/medical-cardiology-qa', endpoint='https://huggingface.co', repo_type='dataset', repo_id='lmassaron/medical-cardiology-qa'), pr_revision=None, pr_num=None)

In [20]:
complete_dataset["train"][5]

{'messages': [{'content': 'You are a knowledgeable medical assistant specializing in cardiology. Answer clinical questions accurately, focusing on diagnostic criteria, treatment guidelines, and pathophysiology.',
   'role': 'system'},
  {'content': 'Can hypertension cause chronic kidney disease?',
   'role': 'user'},
  {'content': 'Yes, long-standing untreated hypertension can cause chronic kidney disease.',
   'role': 'assistant'}]}